# Курсов проект по дисциплината
„Планиране на експеримента“

## 1. Данни

Контекст: Фотоволтаична система - показател за ефективност - подвариант A

Фактор z_t: Слънчева радиация

Файлове за работа: student_data/V49_raw.csv и student_data/V49_future.csv

Структура на входните файлове: Vxx_raw.csv съдържа t, x_norm, z_factor, y_raw; Vxx_future.csv съдържа t, x_norm, z_factor.

Размер и особености на данните: 120 наблюдения за t = 1..120, хоризонт за прогноза t = 121..126, 4 липсващи стойности и 3 аномални точки в наблюдавания сигнал.

Важно ограничение: използвайте само student_data файловете за Вашия вариант. Не използвайте instructor_data, truth файлове или предварително известни параметри на генератора.

Специален акцент за Вашия вариант: В етап 5 изрично коментирайте риска от overfitting при M2 спрямо M1.

## 2. Общата задача

Да се изгради напълно възпроизводим програмен проект, който започва от RAW данни, преминава през първична обработка, статистически анализ, моделиране и прогноза, и завършва с оптимизация на параметрите на избрания модел. Всички етапи трябва да бъдат реализирани с код; не се допуска решение, основано само на ръчна обработка в Excel или LibreOffice.

## Как да използвате този шаблон

Попълвайте TODO секциите с Вашия код и кратки пояснения. Не изтривайте ключовите клетки за импорти, зареждане на данните и финални проверки. Добра практика е след всяка по-съществена стъпка да добавяте кратки интерпретации в Markdown клетка.

In [4]:
# Основни библиотеки
import sys
import json
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
plt.rcParams["figure.figsize"] = (10, 4)

print("Python:", sys.version.split()[0])
print("NumPy :", np.__version__)
print("Pandas:", pd.__version__)


Python: 3.12.13
NumPy : 2.0.2
Pandas: 2.2.2


In [5]:
# Помощни функции
def find_file(filename: str) -> Path:
    search_roots = [Path.cwd(), Path.cwd().parent]
    candidates = []
    for root in search_roots:
        candidates.extend([
            root / "student_data" / filename,
            root / filename,
            root / "data" / filename,
        ])
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f"Не откривам файла: {filename}. Проверете структурата на проекта.")


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


def bias(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(y_pred - y_true)


def empirical_edf(x):
    x = np.sort(np.asarray(x, dtype=float))
    p = np.arange(1, len(x) + 1) / len(x)
    return x, p


def ols_fit(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_hat = X @ beta
    residuals = y - y_hat
    return beta, y_hat, residuals


def standardize_from_train(train_series, full_series):
    mu = float(np.mean(train_series))
    sd = float(np.std(train_series, ddof=1))
    if sd == 0:
        raise ValueError("Стандартното отклонение е 0. Проверете регресора.")
    return (full_series - mu) / sd, mu, sd


def project_box(theta, lower, upper):
    theta = np.asarray(theta, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    return np.minimum(np.maximum(theta, lower), upper)


def fit_ar1_ols(residuals):
    r = np.asarray(residuals, dtype=float)
    r_prev = r[:-1]
    r_next = r[1:]
    phi = np.dot(r_prev, r_next) / np.dot(r_prev, r_prev)
    eta = r_next - phi * r_prev
    sigma_eta = np.std(eta, ddof=1)
    return phi, sigma_eta


In [6]:
ASSIGNMENT = {
    "variant_code": "V49",
    "scenario_title": "Photovoltaic System Performance",
    "factor_label": "Solar Radiation",
    "raw_file": "V49_raw.csv",
    "future_file": "V49_future.csv",
}

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(".")

RAW_FILE = PROJECT_ROOT / "student_data" / ASSIGNMENT["raw_file"]
FUTURE_FILE = PROJECT_ROOT / "student_data" / ASSIGNMENT["future_file"]

RESULTS_DIR = PROJECT_ROOT / "results"
GRAPHICS_DIR = RESULTS_DIR / "graphics"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(RAW_FILE)
future_df = pd.read_csv(FUTURE_FILE)

display(raw_df.head())
display(future_df.head())

print("RAW shape:", raw_df.shape)
print("FUTURE shape:", future_df.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'student_data/V49_raw.csv'

## Етап 0. Подготовка на проекта

Задание: Създайте проектова структура, импортирайте данните, проверете типовете на колоните, установете броя на редовете и запазете непроменено копие на RAW данните.

Документирайте с каква команда се стартира проектът.

Опишете използваните библиотеки и версията на езика/средата в README.txt.

Подгответе папки поне source/ и results/ за кода и резултатите.

In [ ]:
# TODO: Потвърдете структурата и типовете на входните данни.
display(raw_df.dtypes.to_frame("dtype"))
display(future_df.dtypes.to_frame("dtype"))

# TODO: Създайте непроменено копие на RAW данните.
raw_original = raw_df.copy(deep=True)

# TODO: Добавете проверки за:
# 1) липсващи стойности по колони
print("Липсващи стойности по колони:")
display(raw_df.isna().sum().to_frame("count"))

# 2) дублирани редове
print("Дублирани редове:", raw_df.duplicated().sum())

# 3) монотонност на t
print("t е монотонно нарастваща:", raw_df["t"].is_monotonic_increasing)

# 4) очакван брой редове
print("RAW shape:", raw_df.shape)
print("FUTURE shape:", future_df.shape)

print("Очаквани RAW = 120:", len(raw_df) == 120)
print("Очаквани FUTURE = 6:", len(future_df) == 6)

# 5) диапазони на x_norm и z_factor
print("\nДиапазон x_norm:", raw_df["x_norm"].min(), "→", raw_df["x_norm"].max())
print("Диапазон z_factor:", raw_df["z_factor"].min(), "→", raw_df["z_factor"].max())

print("Липсващи стойности в RAW:")
display(raw_df.isna().sum().to_frame("count"))
print("Дублирани редове:", raw_df.duplicated().sum())
print("t е монотонно нарастваща колона:", raw_df["t"].is_monotonic_increasing)

# TODO: Опишете в README.txt как се стартира проектът и каква е структурата на папките.


## Етап 1. Първична обработка на данни

Задание: Постройте графика y_raw(t), открийте липсващите стойности и аномалните точки, сравнете поне един локален филтър от лекцията и още един допълнителен критерий по избор, интерполирайте липсващите стойности и постройте изгладен ред y_smooth(t).

Сравнете поне два локални подхода, например F(1,1), F(2,1) или F(2,2), и аргументирайте избора на финален CLEAN pipeline.

Създайте CLEAN.csv с най-малко следните колони: t, x_norm, z_factor, y_raw, y_clean, is_missing, is_outlier. Препоръчително е да добавите и y_smooth.

Покажете графика с поне raw, y_clean и y_smooth върху една и съща времева ос.

In [ ]:
# Работно копие
df = raw_df.copy()

# TODO 1: маркирайте липсващите стойности
df["is_missing"] = df["y_raw"].isna()

# TODO 2: реализирайте критерий(и) за аномални точки.
# Примерни идеи:
# - локални разлики / втори разлики
# - rolling median / rolling MAD
# - сравнение на F(1,1), F(2,1), F(2,2)

def local_median_predictor(y):
    y = np.asarray(y, dtype=float)
    n = len(y)
    y_hat = np.full(n, np.nan)

    for i in range(2, n-2):
        neigh = [y[i-2], y[i-1], y[i+1], y[i+2]]
        if all(np.isfinite(v) for v in neigh):
            y_hat[i] = np.median(neigh)
    return y_hat


def mad(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return 0
    med = np.median(x)
    return 1.4826 * np.median(np.abs(x - med))


# работим върху копие без NaN (за детекция)
y = df["y_raw"].values.astype(float)

# временна интерполация за стабилност
y_temp = pd.Series(y).interpolate().values

y_hat = local_median_predictor(y_temp)
res = y_temp - y_hat

scale = mad(res)
thr = 6 * scale  # консервативен праг

df["is_outlier"] = np.abs(res) > thr

def smooth_f11(y):
    """
    Филтър F(1,1)
    Равномерно локално осредняване:
    [1, 1, 1] / 3
    """

    y = np.asarray(y, dtype=float)
    s = y.copy()

    for i in range(1, len(y)-1):
        s[i] = (
            y[i-1] +
            y[i] +
            y[i+1]
        ) / 3

    return s


def smooth_f21(y):
    """
    Филтър F(2,1)
    По-силно изглаждане:
    [1, 2, 3, 2, 1] / 9
    """

    y = np.asarray(y, dtype=float)
    s = y.copy()

    for i in range(2, len(y)-2):
        s[i] = (
            y[i-2] +
            2*y[i-1] +
            3*y[i] +
            2*y[i+1] +
            y[i+2]
        ) / 9

    return s


def smooth_f22(y):
    """
    Филтър F(2,2)
    Още по-гладък локален филтър:
    [1, 2, 2, 2, 1] / 8
    """

    y = np.asarray(y, dtype=float)
    s = y.copy()

    for i in range(2, len(y)-2):
        s[i] = (
            y[i-2] +
            2*y[i-1] +
            2*y[i] +
            2*y[i+1] +
            y[i+2]
        ) / 8

    return s

# TODO 3: интерполирайте липсващите стойности -> y_interp

df["y_interp"] = df["y_raw"].copy()

mask = np.isfinite(df["y_interp"])

df.loc[~mask, "y_interp"] = np.interp(
    df.loc[~mask, "t"],
    df.loc[mask, "t"],
    df.loc[mask, "y_interp"]
)

# TODO 4: коригирайте outlier-и -> y_clean

y_clean = df["y_interp"].copy()

out_idx = df["is_outlier"].values

# маркираме outlier-и като липсващи
y_clean[out_idx] = np.nan

# локална линейна интерполация
y_clean = pd.Series(y_clean).interpolate().values

df["y_clean"] = y_clean

# TODO 5: приложете локални филтри

df["smooth_f11"] = smooth_f11(df["y_clean"])
df["smooth_f21"] = smooth_f21(df["y_clean"])
df["smooth_f22"] = smooth_f22(df["y_clean"])


# Избираме финален smooth сигнал
df["y_smooth"] = df["smooth_f21"]

# TODO 6: CLEAN.csv

df_final = df[[
    "t",
    "x_norm",
    "z_factor",
    "y_raw",
    "y_clean",
    "is_missing",
    "is_outlier",
    "smooth_f11",
    "smooth_f21",
    "smooth_f22",
    "y_smooth"
]]

df_final.to_csv(RESULTS_DIR / "CLEAN.csv", index=False)

# TODO 7: визуализация

# ГРАФИКА 1: RAW SIGNAL

plt.figure(figsize=(12, 4))

plt.plot(
    df["t"],
    df["y_raw"],
    linewidth=1.8,
    label="y_raw"
)

# missing values
plt.scatter(
    df.loc[df["is_missing"], "t"],
    [df["y_raw"].mean()] * df["is_missing"].sum(),
    color="black",
    marker="x",
    s=70,
    label="missing"
)

# outliers
plt.scatter(
    df.loc[df["is_outlier"], "t"],
    df.loc[df["is_outlier"], "y_raw"],
    color="red",
    s=60,
    label="outliers",
    zorder=5
)

plt.xlabel("t")
plt.ylabel("y_raw")

plt.title("RAW сигнал y_raw(t)")

plt.legend()
plt.grid(True)

plt.savefig(
    GRAPHICS_DIR / "fig1_raw_signal.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ГРАФИКА 2: Сравнение на локални филтри

plt.figure(figsize=(12, 4))

plt.plot(df["t"],
         df["y_clean"],
         label="y_clean",
         linewidth=2)

plt.plot(df["t"],
         df["smooth_f11"],
         label="F(1,1)")

plt.plot(df["t"],
         df["smooth_f21"],
         label="F(2,1)")

plt.plot(df["t"],
         df["smooth_f22"],
         label="F(2,2)")

plt.xlabel("t")
plt.ylabel("Signal")

plt.title("Сравнение на локални филтри")

plt.legend()
plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig2_filters_comparison.png", dpi=300, bbox_inches="tight")

plt.show()

# ГРАФИКА 3: RAW vs CLEAN vs SMOOTH

plt.figure(figsize=(12, 5))

# RAW SIGNAL
plt.scatter(
    df["t"],
    df["y_raw"],
    label="y_raw",
    alpha=0.5,
    s=18
)

# CLEAN SIGNAL
plt.plot(
    df["t"],
    df["y_clean"],
    label="y_clean",
    linewidth=2,
    alpha=0.8
)

# SMOOTH SIGNAL
plt.plot(
    df["t"],
    df["y_smooth"],
    label="y_smooth",
    linewidth=3
)

plt.xlabel("t")
plt.ylabel("Signal")

plt.title("RAW, CLEAN и SMOOTH сигнал")

plt.legend()
plt.grid(True)

plt.savefig(
    GRAPHICS_DIR / "fig3_cleaning_pipeline.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Изследвани са три локални метода за изглаждане:

*   F(1,1) — най-слабо изглаждане; запазва локалните колебания, но оставя повече шум;
*   F(2,1) — балансирано изглаждане с добро потискане на шума;
*   F(2,2) — най-силно изглаждане; потиска шума ефективно, но редуцира локалните пикове.
#
Като финален CLEAN pipeline е избран F(2,1), тъй като:
*   премахва голяма част от високочестотния шум;
*   запазва основния тренд;
*   не деформира съществено динамиката на сигнала.





## Етап 2. Случайни величини и разпределения

Задание: Дефинирайте случаен компонент e_t = y_clean(t) - y_smooth(t). Оценете средна стойност, дисперсия, стандартно отклонение, медиана, квартилите и коефициент на вариация.

Постройте хистограма и емпирична функция на разпределение за e_t.

Сравнете e_t с нормално разпределение N(m_hat, s_hat^2). Не е нужно формален тест, но е нужен аргументиран коментар.

Изведете кратък извод дали допускането за нормалност е разумно за Вашите данни.

In [ ]:
# TODO: уверете се, че df съдържа y_clean и y_smooth
e_t = df["y_clean"] - df["y_smooth"]

# TODO: изчислете mean, var, std, median, Q1, Q3, CV
mean_e = float(np.mean(e_t))
var_e = float(np.var(e_t, ddof=1))
std_e = float(np.std(e_t, ddof=1))

median_e = float(np.median(e_t))

q1_e = float(np.quantile(e_t, 0.25))
q3_e = float(np.quantile(e_t, 0.75))

# коефициент на вариация
cv_e = std_e / (abs(mean_e) + 1e-12)

stats_e = {
    "mean": mean_e,
    "variance": var_e,
    "std": std_e,
    "median": median_e,
    "Q1": q1_e,
    "Q3": q3_e,
    "CV": cv_e
}

stats_df = pd.DataFrame(stats_e, index=["e_t"])

display(stats_df.round(6))

# TODO: построете histogram и empirical EDF за e_t

# Хистограма

plt.figure(figsize=(10, 4))

plt.hist(e_t,
         bins=20,
         density=True)

plt.xlabel("e_t")
plt.ylabel("Density")

plt.title("Хистограма на случайния компонент e_t")

plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig4_histogram.png", dpi=300, bbox_inches="tight")

plt.show()

# EDF

x_edf, p_edf = empirical_edf(e_t)

plt.figure(figsize=(10, 4))

plt.step(x_edf,
         p_edf,
         where="post")

plt.xlabel("e_t")
plt.ylabel("F_n(x)")

plt.title("Емпирична функция на разпределение")

plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig5_EDF.png", dpi=300, bbox_inches="tight")

plt.show()

# TODO: сравнете визуално e_t с N(m_hat, s_hat^2) и направете кратък извод

mu_hat = mean_e
sigma_hat = std_e

xs = np.linspace(e_t.min(), e_t.max(), 400)

normal_pdf = (
    1 / (sigma_hat * np.sqrt(2*np.pi))
) * np.exp(
    -0.5 * ((xs - mu_hat) / sigma_hat)**2
)

plt.figure(figsize=(10, 4))

plt.hist(e_t,
         bins=20,
         density=True,
         alpha=0.7,
         label="Empirical")

plt.plot(xs,
         normal_pdf,
         linewidth=2,
         label="Normal PDF")

plt.xlabel("e_t")
plt.ylabel("Density")

plt.title("Хистограма на e_t и теоретична нормална плътност")

plt.legend()
plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig6_normal_distribution_comparison.png", dpi=300, bbox_inches="tight")

plt.show()


**Анализ на случайния компонент et**
	​


Случайният компонент е дефиниран като:

et = y clean(t)− y smooth(t)

Получените стойности на средната стойност са близки до нула, което показва, че изглаждането не въвежда систематично отклонение.

Хистограмата на et има приблизително симетрична форма около нулата. Емпиричната функция на разпределение също е близка до тази на нормално разпределение.

При визуално сравнение с теоретично нормално разпределение N(m_hat, s_hat^2) се наблюдава добро съответствие в централната област, с възможни малки отклонения в опашките.

Следователно допускането за приблизителна нормалност на случайния компонент е разумно и може да се използва при следващите етапи на моделиране.

## Етап 3. Двумерни случайни величини, ковариация и корелация

Задание: Разгледайте двойките (z_t, y_clean(t)) за t = 1..120. Изчислете извадкова ковариация и коефициент на корелация на Пирсън.

Постройте scatter plot на z_t спрямо y_clean.

Коментирайте дали зависимостта изглежда линейна, силна, слаба или потенциално нелинейна.

Не се ограничавайте само до числената стойност на r; тълкувайте резултата в контекста на варианта.

In [ ]:
# TODO: използвайте двойките (z_t, y_clean(t))

z = df["z_factor"].to_numpy()
y = df["y_clean"].to_numpy()

# TODO: изчислете sample covariance и Pearson correlation

cov_zy = float(np.cov(z, y, ddof=1)[0, 1])

corr_zy = float(np.corrcoef(z, y)[0, 1])

stats_zy = {
    "covariance": cov_zy,
    "pearson_r": corr_zy
}

stats_df = pd.DataFrame(stats_zy, index=["z vs y_clean"])

display(stats_df.round(6))

# TODO: scatter plot и коментар за линейност / нелинейност

plt.figure(figsize=(8, 5))

plt.scatter(
    df["z_factor"],
    df["y_clean"],
    s=20,
    alpha=0.7
)

plt.xlabel("z_factor")
plt.ylabel("y_clean")

plt.title("Scatter plot: z_factor vs y_clean")

plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig7_scatter_plot.png", dpi=300, bbox_inches="tight")

plt.show()

Наблюдава се много силна положителна линейна зависимост между z_factor и y_clean. Стойността на коефициента на Пирсън (r ≈ 0.96) е близка до 1, което показва, че връзката между двете променливи е почти линейна.

Scatter plot-ът потвърждава тази зависимост — точките са подредени около ясно изразена възходяща права линия. Това означава, че с увеличаване на z_factor стойностите на y_clean също нарастват почти пропорционално.

Ковариацията също е положителна и голяма, което допълнително потвърждава, че променливите се изменят в една и съща посока.

## Етап 4. Данни от извадка - групирани и негрупирани оценки

Задание: Третирайте y_clean като извадка от генерална съвкупност. Изчислете статистиките първо върху негрупирани данни, а след това върху групирани данни с 8 равни интервала.

Сравнете средна стойност, медиана, дисперсия, IQR и коефициент на вариация за двата подхода.

Покажете таблица grouped срещу ungrouped.

Коментирайте каква информация се губи при групиране на извадката.

In [ ]:
# TODO: негрупирани статистики за y_clean
y = df["y_clean"].dropna().values

mean_u = float(np.mean(y))
median_u = float(np.median(y))
var_u = float(np.var(y, ddof=1))

q1_u = float(np.quantile(y, 0.25))
q3_u = float(np.quantile(y, 0.75))
iqr_u = q3_u - q1_u

cv_u = float(np.std(y, ddof=1) / (abs(mean_u) + 1e-12))


ungrouped = {
    "mean": mean_u,
    "median": median_u,
    "variance": var_u,
    "IQR": iqr_u,
    "CV": cv_u
}


# TODO: групирайте y_clean в 8 равни интервала

bins = np.linspace(df["y_clean"].min(), df["y_clean"].max(), 9)
df["bin"] = pd.cut(df["y_clean"], bins=bins, include_lowest=True)

group_stats = df.groupby("bin")["y_clean"]

# TODO: оценете grouped статистики

counts = group_stats.count()
centers = group_stats.mean()

mean_g = float(np.sum(centers * counts) / np.sum(counts))

# медиана (приближена чрез групирани кумулативни честоти)
cdf = counts.cumsum() / counts.sum()
median_bin_idx = np.searchsorted(cdf.values, 0.5)
median_g = float(centers.iloc[median_bin_idx])

# дисперсия (приближена)
var_g = float(np.sum(counts * (centers - mean_g) ** 2) / np.sum(counts))

q1_g = float(np.quantile(centers, 0.25))
q3_g = float(np.quantile(centers, 0.75))
iqr_g = q3_g - q1_g

cv_g = float(np.sqrt(var_g) / (abs(mean_g) + 1e-12))

grouped = {
    "mean": mean_g,
    "median": median_g,
    "variance": var_g,
    "IQR": iqr_g,
    "CV": cv_g
}

grouped_df = pd.DataFrame([grouped])

# TODO: създайте сравнителна таблица grouped vs ungrouped
comparison_df = pd.DataFrame({
    "ungrouped": ungrouped,
    "grouped(8 bins)": grouped
})

display(comparison_df)

comparison_df.to_csv(RESULTS_DIR / "stats_comparison.csv", index=True)

При сравнение между негрупирани и групирани данни се вижда, че средната стойност остава идентична, което е очаквано, тъй като групирането не променя общото централно ниво на извадката.

Медианата при групираните данни се измества (120.30 → 117.12), което е резултат от загубата на индивидуална информация и замяната на реалните стойности с интервални представители. Това показва, че медианата е чувствителна към начина на дискретизация.

Дисперсията намалява леко (185.00 → 179.78), което е типично при групиране, тъй като вътрешногруповата вариация се изглажда и част от разсейването се губи.

IQR остава близък, но леко се променя (22.45 → 23.04), което показва, че разпределението в централната част се запазва сравнително добре, но границите вече са по-грубо представени.

Коефициентът на вариация също леко намалява (0.115 → 0.113), което показва леко подценяване на относителната вариабилност при групиране.

## Етап 5. Моделиране и оценка на грешката

Задание: Използвайте разделяне train/test с train = t = 1..90 и test = t = 91..120. Стандартизирайте регресорите по train частта:

u_t = (x_norm - mean_train(x_norm)) / std_train(x_norm),
v_t = (z_factor - mean_train(z_factor)) / std_train(z_factor)

Напаснете с МНМК следните два модела:

M1: y_t = beta0 + beta1*u_t + beta2*v_t + eps_t
M2: y_t = beta0 + beta1*u_t + beta2*v_t + beta3*v_t^2 + eps_t

Изчислете MAE, RMSE и Bias върху train и test.

Изберете по-добрия модел и защитете избора си с числа и кратък коментар.

Покажете поне една графика на напасването и една таблица с оценените параметри.

In [ ]:
# Подгответе CLEAN данните за моделиране
clean_df = df.copy()

train_df = clean_df[clean_df["t"] <= 90].copy()
test_df = clean_df[(clean_df["t"] >= 91) & (clean_df["t"] <= 120)].copy()

# TODO: стандартизация само по train частта

clean_df["u_t"], x_mean, x_std = standardize_from_train(
    train_df["x_norm"],
    clean_df["x_norm"]
)

clean_df["v_t"], z_mean, z_std = standardize_from_train(
    train_df["z_factor"],
    clean_df["z_factor"]
)

# TODO: обновете train_df и test_df след добавяне на u_t и v_t

train_df = clean_df[clean_df["t"] <= 90].copy()
test_df = clean_df[clean_df["t"] >= 91].copy()

# TARGET

y_train = train_df["y_clean"].to_numpy()
y_test = test_df["y_clean"].to_numpy()

# MODEL M1
# y = beta0 + beta1*u_t + beta2*v_t


X_train_m1 = np.column_stack([
    np.ones(len(train_df)),
    train_df["u_t"],
    train_df["v_t"]
])

X_test_m1 = np.column_stack([
    np.ones(len(test_df)),
    test_df["u_t"],
    test_df["v_t"]
])

beta_m1, yhat_train_m1, resid_train_m1 = ols_fit(
    X_train_m1,
    y_train
)

yhat_test_m1 = X_test_m1 @ beta_m1

# ГРЕШКИ M1

metrics_m1 = {
    "train_MAE": mae(y_train, yhat_train_m1),
    "test_MAE": mae(y_test, yhat_test_m1),

    "train_RMSE": rmse(y_train, yhat_train_m1),
    "test_RMSE": rmse(y_test, yhat_test_m1),

    "train_Bias": bias(y_train, yhat_train_m1),
    "test_Bias": bias(y_test, yhat_test_m1)
}

# MODEL M2
# y = beta0 + beta1*u_t + beta2*v_t + beta3*v_t^2

X_train_m2 = np.column_stack([
    np.ones(len(train_df)),
    train_df["u_t"],
    train_df["v_t"],
    train_df["v_t"]**2
])

X_test_m2 = np.column_stack([
    np.ones(len(test_df)),
    test_df["u_t"],
    test_df["v_t"],
    test_df["v_t"]**2
])

beta_m2, yhat_train_m2, resid_train_m2 = ols_fit(
    X_train_m2,
    y_train
)

yhat_test_m2 = X_test_m2 @ beta_m2

# ГРЕШКИ M2

metrics_m2 = {
    "train_MAE": mae(y_train, yhat_train_m2),
    "test_MAE": mae(y_test, yhat_test_m2),

    "train_RMSE": rmse(y_train, yhat_train_m2),
    "test_RMSE": rmse(y_test, yhat_test_m2),

    "train_Bias": bias(y_train, yhat_train_m2),
    "test_Bias": bias(y_test, yhat_test_m2)
}

# ТАБЛИЦА С ГРЕШКИТЕ

metrics_df = pd.DataFrame(
    [metrics_m1, metrics_m2],
    index=["M1", "M2"]
)

display(metrics_df.round(6))

metrics_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=True)

# ТАБЛИЦА С ПАРАМЕТРИТЕ

params_df = pd.DataFrame({
    "M1": [
        beta_m1[0],
        beta_m1[1],
        beta_m1[2],
        np.nan
    ],
    "M2": [
        beta_m2[0],
        beta_m2[1],
        beta_m2[2],
        beta_m2[3]
    ]
}, index=[
    "beta0",
    "beta1",
    "beta2",
    "beta3"
])

display(params_df.round(6))

# ИЗБОР НА МОДЕЛ

print("M1 test RMSE:", metrics_m1["test_RMSE"])
print("M2 test RMSE:", metrics_m2["test_RMSE"])

if metrics_m2["test_RMSE"] < metrics_m1["test_RMSE"]:
    best_model = "M2"
else:
    best_model = "M1"

print("\nИзбран модел:", best_model)

# ГРАФИКА НА НАПАСВАНЕТО

plt.figure(figsize=(12, 5))

plt.plot(
    train_df["t"],
    y_train,
    label="Observed train",
    linewidth=2
)

plt.plot(
    train_df["t"],
    yhat_train_m1,
    label="M1 fit",
    linewidth=2
)

plt.plot(
    train_df["t"],
    yhat_train_m2,
    label="M2 fit",
    linewidth=2
)

plt.axvline(
    90,
    color="black",
    linestyle="--",
    alpha=0.7
)

plt.xlabel("t")
plt.ylabel("y_clean")

plt.title("Model fit върху train частта")

plt.legend()
plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig8_model_fit.png", dpi=300, bbox_inches="tight")

plt.show()


Моделите бяха оценени чрез разделяне train/test:

train: t = 1..90
test: t = 91..120

Използвани бяха следните модели:

M1 : yt = β0 + β1 * ut + β2 * vt + εt

M2 : yt = β0 + β1 * ut + β2 * vt + β3 * vt2 + εt
#
Получените грешки показват:

```
Модел	  Train RMSE	  Test RMSE
M1	       3.5306	      3.7850
M2	       3.5241	      3.8168
```

Модел M2 има минимално по-добро напасване върху train частта, но по-лош резултат върху test данните. Това показва, че квадратичният член започва да напасва част от шума в train извадката и не подобрява способността за обобщение върху нови данни.

Следователно M2 показва риск от overfitting.

Поради по-ниската test RMSE и по-простата структура, за финален модел е избран M1.

## Етап 6. Стохастично прогнозиране

Задание: За избрания модел дефинирайте остатъци r_t = y_clean(t) - y_hat(t) върху train частта. Оценете AR(1) модел r_t = phi*r_(t-1) + eta_t с метод на най-малките квадрати и използвайте V49_future.csv за прогноза на y_t за t = 121..126.

Представете точкова прогноза и приблизителен 95% прогнозен интервал.

Създайте forecast.csv с най-малко колони: t, x_norm, z_factor, y_forecast, lower95, upper95.

Покажете графика на прогнозата заедно с последната част на наблюдавания ред.

In [ ]:
# TODO: изчислете остатъците върху train за избрания модел
residuals_train = resid_train_m1.copy()

# TODO: оценете AR(1) модел върху residuals_train
phi, sigma_eta = fit_ar1_ols(residuals_train)

print("phi =", round(phi, 6))
print("sigma_eta =", round(sigma_eta, 6))

# TODO: стандартизирайте future_df с train mean/std
future_df["u_t"] = (
    future_df["x_norm"] - x_mean
) / x_std

future_df["v_t"] = (
    future_df["z_factor"] - z_mean
) / z_std

# DESIGN MATRIX ЗА M1

X_future = np.column_stack([
    np.ones(len(future_df)),
    future_df["u_t"],
    future_df["v_t"]
])

# БАЗОВА ПРОГНОЗА ОТ M1

y_base = X_future @ beta_m1

# AR(1) ПРОГНОЗА НА ОСТАТЪЦИТЕ

last_resid = residuals_train[-1]

future_residuals = []

r_prev = last_resid

for h in range(len(future_df)):

    r_next = phi * r_prev

    future_residuals.append(r_next)

    r_prev = r_next

future_residuals = np.array(future_residuals)

# ФИНАЛНА ПРОГНОЗА

y_forecast = y_base + future_residuals

# 95% ПРОГНОЗЕН ИНТЕРВАЛ

lower95 = y_forecast - 1.96 * sigma_eta
upper95 = y_forecast + 1.96 * sigma_eta

# FORECAST DATAFRAME

forecast_df = pd.DataFrame({
    "t": future_df["t"],
    "x_norm": future_df["x_norm"],
    "z_factor": future_df["z_factor"],
    "y_forecast": y_forecast,
    "lower95": lower95,
    "upper95": upper95
})

forecast_df.to_csv(RESULTS_DIR / "forecast.csv", index=False)

display(forecast_df.round(4))

# SAVE forecast.csv

forecast_path = RESULTS_DIR / "forecast.csv"

forecast_df.to_csv(forecast_path, index=False)

print("Saved to:", forecast_path)

print("forecast.csv saved successfully.")

# ГРАФИКА НА ПРОГНОЗАТА

plt.figure(figsize=(12, 5))

# последни наблюдения
plt.plot(
    clean_df["t"].tail(30),
    clean_df["y_clean"].tail(30),
    label="Observed",
    linewidth=2
)

# прогноза
plt.plot(
    forecast_df["t"],
    forecast_df["y_forecast"],
    marker="o",
    linewidth=2,
    label="Forecast"
)

# доверителен интервал
plt.fill_between(
    forecast_df["t"],
    forecast_df["lower95"],
    forecast_df["upper95"],
    alpha=0.3,
    label="95% interval"
)

plt.xlabel("t")
plt.ylabel("y")

plt.title("Forecast for t = 121..126")

plt.legend()
plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig9_forecast.png", dpi=300, bbox_inches="tight")

plt.show()

За избрания модел M1 бяха изчислени остатъците:

rt​ = yclean(t) − y^(t)

върху train периода t=1..90. Върху тях беше оценен AR(1) модел:

rt = ϕrt − 1 + ηt

чрез метода на най-малките квадрати.

Получената оценка е:

ϕ = 0.274851

което показва слаба положителна автокорелация на остатъците. Тъй като ∣ϕ∣<1, процесът е стабилен и не показва експлозивно поведение.

Оцененото стандартно отклонение на шума е:

ση = 3.414011

На база на избрания модел и бъдещите стойности на регресорите от V49_future.csv бяха генерирани прогнози за периода t = 121..126, както и приблизителни 95% прогнозни интервали:

y^t ± 1.96ση

Получените прогнози показват лек спад до t=124, последван от възстановяване и покачване към t=126. Това поведение е в съответствие с изменението на фактора „слънчева радиация“ (z_factor) и е физически смислено за фотоволтаична система.

Ширината на прогнозните интервали е умерена, което показва сравнително стабилна прогноза и ограничена неопределеност на модела.

## Етап 7. Градиентни / квазиградиентни методи

Задание: Преоценете параметрите на избрания модел чрез projected gradient върху критерия J(theta) = mean((y - y_hat)^2) в train частта.

Използвайте стартова точка theta^(0) = (mean(y_train), 0, 0, 0).

Наложете ограничения: beta0 in [mean(y_train)-30, mean(y_train)+30], beta1, beta2, beta3 in [-20, 20].

Използвайте стъпка rho_k = 1/(k+1). Стоп критерий: ||theta^(k+1)-theta^(k)||_2 < 1e-6 или k = 5000.

Покажете графика на J(theta_k) спрямо k и сравнете получения резултат с МНМК. Ако е избран M1, игнорирайте beta3.

In [ ]:
# Използваме M1
X_pg = X_train_m1.copy()
y_pg = y_train.copy()

# Целева функция J(theta)

def mse_objective(theta, X, y):

    y_hat = X @ theta

    return np.mean((y - y_hat) ** 2)

# Аналитичен градиент

def mse_gradient(theta, X, y):

    residual = X @ theta - y

    grad = (2 / len(y)) * (X.T @ residual)

    return grad

# Projected Gradient Method

def projected_gradient(
    X,
    y,
    theta0,
    lower,
    upper,
    max_iter=5000,
    tol=1e-6
):

    theta = np.asarray(theta0, dtype=float).copy()

    history = []

    for k in range(max_iter):

        rho_k = 1.0 / (k + 1)

        grad = mse_gradient(theta, X, y)

        theta_new = project_box(
            theta - rho_k * grad,
            lower,
            upper
        )

        J_new = mse_objective(theta_new, X, y)

        history.append(J_new)

        if np.linalg.norm(theta_new - theta) < tol:

            theta = theta_new

            print("Converged at iteration:", k + 1)

            break

        theta = theta_new

    return theta, history

# Стартова точка theta^(0)

mean_y_train = np.mean(y_train)

theta0 = np.array([
    mean_y_train,
    0,
    0
])

# Ограничения

lower = np.array([
    mean_y_train - 30,
    -20,
    -20
])

upper = np.array([
    mean_y_train + 30,
    20,
    20
])

# Стартиране

theta_pg, history_pg = projected_gradient(
    X_pg,
    y_pg,
    theta0,
    lower,
    upper
)

print("\nProjected Gradient parameters:")
print(theta_pg)

# OLS параметри за сравнение

print("\nOLS parameters:")
print(beta_m1)

# Train prediction

yhat_pg = X_pg @ theta_pg

# Метрики

mae_pg = mae(y_train, yhat_pg)
rmse_pg = rmse(y_train, yhat_pg)
bias_pg = bias(y_train, yhat_pg)

print("\nProjected Gradient metrics:")

print("MAE :", mae_pg)
print("RMSE:", rmse_pg)
print("Bias:", bias_pg)

# Сравнителна таблица

compare_df = pd.DataFrame({
    "OLS_M1": beta_m1,
    "Projected_Gradient": theta_pg
},
index=["beta0", "beta1", "beta2"])

display(compare_df.round(6))

# Графика на сходимостта

plt.figure(figsize=(10, 4))

plt.plot(history_pg)

plt.xlabel("Iteration k")
plt.ylabel("J(theta_k)")

plt.title("Projected Gradient Convergence")

plt.grid(True)

plt.savefig(GRAPHICS_DIR / "fig10_gradient.png", dpi=300, bbox_inches="tight")

plt.show()

Параметрите на избрания модел M1 бяха преоценени чрез projected gradient method върху train частта, използвайки критерия:

J(θ)=mean((y−Xθ)
2
)

Използвана беше стартова точка:

θ
(0)
=(
y
	​

train
	​

,0,0)

и ограничения:

β
0
	​

∈[
y
	​

train
	​

−30,
y
	​

train
	​

+30]
β
1
	​

,β
2
	​

∈[−20,20]

Стъпката беше зададена като:

ρ
k
	​

=
k+1
1
	​


Алгоритъмът достигна сходимост след 178 итерации при зададения критерий:

∥θ
(k+1)
−θ
(k)
∥
2
	​

<10
−6

Получените параметри са:

Параметър	OLS	Projected Gradient
beta0	117.899151	117.899151
beta1	1.329417	1.329494
beta2	13.182422	13.182483

Разликите между параметрите, оценени чрез МНМК и projected gradient, са минимални. Това показва, че оптимизационният алгоритъм успешно достига практически същото решение като аналитичния OLS метод.

Получените метрики също съвпадат почти напълно:

MAE ≈ 2.774
RMSE ≈ 3.531
Bias ≈ 0

Графиката на J(θ
k
	​

) показва гладка монотонна сходимост без нестабилности, което потвърждава коректната работа на алгоритъма.

## 4. Задължителни изходни файлове и материали

Архивът за предаване трябва да се казва 471223002_V49_PE.zip.

Архивът трябва да съдържа поне: report.pdf, README.txt, source/, results/, CLEAN.csv и forecast.csv.

report.pdf трябва да е 8-15 страници без приложения и да съдържа кратка теория, метод, резултати и изводи.

README.txt трябва да съдържа команда за стартиране и кратко описание на файловете.

results/ трябва да съдържа графики и таблици, използвани в отчета.

## 5. Минимален набор графики и таблици

raw графика на y_raw(t)

cleaned vs smooth графика

хистограма и EDF на e_t

scatter plot на z_t срещу y_clean

таблица grouped срещу ungrouped статистики

графика или таблица за model fit и грешки

графика на прогнозата с 95% интервал

графика на сходимостта на projected gradient

## 6. Програмни изисквания

Решението трябва да е напълно програмно и възпроизводимо.

Допускат се Python, MATLAB, R, Julia, C/C++ или друг език, но целият pipeline трябва да може да се изпълни с код.

Не се допуска ръчна обработка като основен метод на решението.

Всички формули, допускания и избрани критерии трябва да бъдат ясно описани в отчета.

## 7. Контролен checklist преди предаване

- [ ] Използвах само файловете за моя вариант.
- [ ] Имам CLEAN.csv и forecast.csv с коректни колони.
- [ ] Сравних M1 и M2 и избрах модел по обоснован начин.
- [ ] Направих прогноза за t = 121..126 и добавих интервал.
- [ ] Изпълних optimization етапа и показах сходимост.
- [ ] README.txt съдържа команда за стартиране.
- [ ] report.pdf съдържа всички графики, таблици и изводи.

In [ ]:
# Финални експорти и бързи проверки
# TODO: разкоментирайте, след като сте подготвили крайните таблици
df_final.to_csv(RESULTS_DIR / "CLEAN.csv", index=False)
forecast_df.to_csv("forecast.csv", index=False)

# TODO: проверка на наличието на задължителните файлове
required = [Path("CLEAN.csv"), Path("forecast.csv")]
for p in required:
    print(f"{p.name}: {'OK' if p.exists() else 'MISSING'}")
